### Compare classification errors across LLM, XGBoost, and Random Forest

This section compares where LLM, XGBoost, and Random Forest make correct or wrong predictions on the same review IDs.

In [50]:
import pandas as pd
from sklearn.metrics import confusion_matrix

In [51]:
# Folder with your prediction files
predictions_dir = "/Users/svetlana/Documents/CAS_ML/Text/Project/text_analysis/generated/predictions"

# Load CSV files
correct_predictions = pd.read_csv(f"{predictions_dir}/correct_predictions_llm.csv")
wrong_predictions = pd.read_csv(f"{predictions_dir}/wrong_predictions_llm.csv")

In [52]:
# Combine correct and wrong predictions
all_predictions = pd.concat(
    [correct_predictions, wrong_predictions],
    ignore_index=True
)

# Get list of all IDs
llm_ids = all_predictions["id"].tolist()


In [53]:
# Load LLM files
correct_predictions_llm = pd.read_csv(
    f"{predictions_dir}/correct_predictions_llm.csv"
)

wrong_predictions_llm = pd.read_csv(
    f"{predictions_dir}/wrong_predictions_llm.csv"
)
# Load Random Forest files
correct_predictions_random_forest = pd.read_csv(
    f"{predictions_dir}/correct_predictions_random_forest.csv"
)

wrong_predictions_random_forest = pd.read_csv(
    f"{predictions_dir}/wrong_predictions_random_forest.csv"
)

# Load XGBoost files
correct_predictions_xgboost = pd.read_csv(
    f"{predictions_dir}/correct_predictions_xgboost.csv"
)

wrong_predictions_xgboost = pd.read_csv(
    f"{predictions_dir}/wrong_predictions_xgboost.csv"
)

In [54]:
# Keep only rows with IDs from LLM sample
correct_predictions_random_forest_filtered = correct_predictions_random_forest[
    correct_predictions_random_forest["id"].isin(llm_ids)
].copy()

wrong_predictions_random_forest_filtered = wrong_predictions_random_forest[
    wrong_predictions_random_forest["id"].isin(llm_ids)
].copy()

correct_predictions_xgboost_filtered = correct_predictions_xgboost[
    correct_predictions_xgboost["id"].isin(llm_ids)
].copy()

wrong_predictions_xgboost_filtered = wrong_predictions_xgboost[
    wrong_predictions_xgboost["id"].isin(llm_ids)
].copy()

In [55]:
print("LLM filtered:", len(correct_predictions_llm))
print("LLM wrong filtered:", len(wrong_predictions_llm))
print("Random Forest correct filtered:", len(correct_predictions_random_forest_filtered))
print("Random Forest wrong filtered:", len(wrong_predictions_random_forest_filtered))
print("XGBoost correct filtered:", len(correct_predictions_xgboost_filtered))
print("XGBoost wrong filtered:", len(wrong_predictions_xgboost_filtered))

LLM filtered: 433
LLM wrong filtered: 59
Random Forest correct filtered: 433
Random Forest wrong filtered: 59
XGBoost correct filtered: 453
XGBoost wrong filtered: 39


In [57]:
import pandas as pd

# Create comparison table
comparison_results = pd.DataFrame({
    "Model": ["LLM", "Random Forest", "XGBoost"],
    "Correct": [
        len(correct_predictions_llm),
        len(correct_predictions_random_forest_filtered),
        len(correct_predictions_xgboost_filtered)
    ],
    "Wrong": [
        len(wrong_predictions_llm),
        len(wrong_predictions_random_forest_filtered),
        len(wrong_predictions_xgboost_filtered)
    ]
})

# Add total and accuracy
comparison_results["Total"] = comparison_results["Correct"] + comparison_results["Wrong"]
comparison_results["Accuracy"] = comparison_results["Correct"] / comparison_results["Total"]
comparison_results = comparison_results.round(2)
comparison_results

,Model,Correct,Wrong,Total,Accuracy
0,LLM,433,59,492,0.88
1,Random Forest,433,59,492,0.88
2,XGBoost,453,39,492,0.92


In [65]:
import pandas as pd

# ============================================================
# Compare LLM, Random Forest, and XGBoost on the same IDs
# ============================================================

# -----------------------------
# 1. Prepare LLM results
# -----------------------------

llm_correct = correct_predictions_llm[["id", "label","text_"]].copy()
llm_correct["llm_correct"] = True

llm_wrong = wrong_predictions_llm[["id", "label", "text_"]].copy()
llm_wrong["llm_correct"] = False

llm_compare = pd.concat(
    [llm_correct, llm_wrong],
    ignore_index=True
)


# -----------------------------
# 2. Prepare Random Forest results
# -----------------------------

rf_correct = correct_predictions_random_forest_filtered[["id"]].copy()
rf_correct["random_forest_correct"] = True

rf_wrong = wrong_predictions_random_forest_filtered[["id"]].copy()
rf_wrong["random_forest_correct"] = False

rf_compare = pd.concat(
    [rf_correct, rf_wrong],
    ignore_index=True
)


# -----------------------------
# 3. Prepare XGBoost results
# -----------------------------

xgb_correct = correct_predictions_xgboost_filtered[["id"]].copy()
xgb_correct["xgboost_correct"] = True

xgb_wrong = wrong_predictions_xgboost_filtered[["id"]].copy()
xgb_wrong["xgboost_correct"] = False

xgb_compare = pd.concat(
    [xgb_correct, xgb_wrong],
    ignore_index=True
)


# -----------------------------
# 4. Merge all models by the same ID
# -----------------------------

comparison_by_id = (
    llm_compare
    .merge(rf_compare, on="id", how="inner")
    .merge(xgb_compare, on="id", how="inner")
)

##comparison_by_id

In [59]:
# Show full text in pandas cells
pd.set_option("display.max_colwidth", None)

# ============================================================
# Analyze agreement and disagreement between LLM and XGBoost
# ============================================================

# Both models correct
both_correct = comparison_by_id[
    comparison_by_id["llm_correct"] &
    comparison_by_id["xgboost_correct"]
].copy()

# Both models wrong
both_wrong = comparison_by_id[
    ~comparison_by_id["llm_correct"] &
    ~comparison_by_id["xgboost_correct"]
].copy()

# Only LLM correct
only_llm_correct = comparison_by_id[
    comparison_by_id["llm_correct"] &
    ~comparison_by_id["xgboost_correct"]
].copy()

# Only XGBoost correct
only_xgboost_correct = comparison_by_id[
    ~comparison_by_id["llm_correct"] &
    comparison_by_id["xgboost_correct"]
].copy()

# Print general counts
print("Same IDs total:", len(comparison_by_id))
print("Both models correct:", len(both_correct))
print("Both models wrong:", len(both_wrong))
print("Only LLM correct:", len(only_llm_correct))
print("Only XGBoost correct:", len(only_xgboost_correct))

# ============================================================
# Label distribution in each error group
# ============================================================

label_error_summary = pd.DataFrame({
    "Both wrong": both_wrong["label"].value_counts(),
    "Only LLM correct": only_llm_correct["label"].value_counts(),
    "Only XGBoost correct": only_xgboost_correct["label"].value_counts()
}).fillna(0).astype(int)

display(label_error_summary)

# ============================================================
# Display rows where both LLM and XGBoost are wrong
# ============================================================

display(both_wrong[["id", "label", "text_"]])

Same IDs total: 492
Both models correct: 401
Both models wrong: 7
Only LLM correct: 32
Only XGBoost correct: 52


,Both wrong,Only LLM correct,Only XGBoost correct
label,,,
CG,2,26,19
OR,5,6,33


,id,label,text_
436,4933,OR,These are refills for the large Silver Stick. It is great that you can get so many reasonably priced.
448,33086,CG,"For the kids who make this game, they love it. We play it almost every day."
452,8057,OR,"works great,considering the price,just download the app (meshare) and enjoy."
453,4853,CG,"These are good ones, I use them at the gym and I keep them for my kids."
454,37571,OR,"They fit perfect, they look expensive, they are the most confortable shoes that i had ever. I love the design"
475,461,OR,This product is sturdy and has a nice design. Will last and last.
480,12735,OR,"I was very pleased with the quality of the video and sound, great concert, the Piano Guys are the best !!!"


In [60]:
# Show full text in pandas cells
pd.set_option("display.max_colwidth", None)

# ============================================================
# Analyze agreement and disagreement between LLM and Random Forest
# ============================================================

# Both models correct
both_correct = comparison_by_id[
    comparison_by_id["llm_correct"] &
    comparison_by_id["random_forest_correct"]
].copy()

# Both models wrong
both_wrong = comparison_by_id[
    ~comparison_by_id["llm_correct"] &
    ~comparison_by_id["random_forest_correct"]
].copy()

# Only LLM correct
only_llm_correct = comparison_by_id[
    comparison_by_id["llm_correct"] &
    ~comparison_by_id["random_forest_correct"]
].copy()

# Only Random Forest correct
only_random_forest_correct = comparison_by_id[
    ~comparison_by_id["llm_correct"] &
    comparison_by_id["random_forest_correct"]
].copy()

# Print general counts
print("Same IDs total:", len(comparison_by_id))
print("Both models correct:", len(both_correct))
print("Both models wrong:", len(both_wrong))
print("Only LLM correct:", len(only_llm_correct))
print("Only Random Forest correct:", len(only_random_forest_correct))

# ============================================================
# Label distribution in each error group
# ============================================================

label_error_summary = pd.DataFrame({
    "Both wrong": both_wrong["label"].value_counts(),
    "Only LLM correct": only_llm_correct["label"].value_counts(),
    "Only Random Forest correct": only_random_forest_correct["label"].value_counts()
}).fillna(0).astype(int)

display(label_error_summary)

# ============================================================
# Display rows where both LLM and Random Forest are wrong
# ============================================================

display(both_wrong[["id", "label", "text_"]])

Same IDs total: 492
Both models correct: 382
Both models wrong: 8
Only LLM correct: 51
Only Random Forest correct: 51


,Both wrong,Only LLM correct,Only Random Forest correct
label,,,
CG,3,45,18
OR,5,6,33


,id,label,text_
448,33086,CG,"For the kids who make this game, they love it. We play it almost every day."
451,450,OR,Love the color. Love that it is lightweight.\nThe bowls are much larger than expected.
453,4853,CG,"These are good ones, I use them at the gym and I keep them for my kids."
454,37571,OR,"They fit perfect, they look expensive, they are the most confortable shoes that i had ever. I love the design"
471,19528,OR,"best price, very large units. too large for cuttlebone holders."
475,461,OR,This product is sturdy and has a nice design. Will last and last.
480,12735,OR,"I was very pleased with the quality of the video and sound, great concert, the Piano Guys are the best !!!"
482,21760,CG,I have boughten a few of these and they are just too big. They are a bit difficult to get on the leash but they are good for the job. They do not slip off the leash when we walk.


In [61]:
both_wrong_ids_comparison = pd.DataFrame({
    "id": comparison_by_id["id"],
    "LLM_XGBoost_wrong": (
        ~comparison_by_id["llm_correct"] &
        ~comparison_by_id["xgboost_correct"]
    ),
    "LLM_RandomForest_wrong": (
        ~comparison_by_id["llm_correct"] &
        ~comparison_by_id["random_forest_correct"]
    ),
    "XGBoost_RandomForest_wrong": (
        ~comparison_by_id["xgboost_correct"] &
        ~comparison_by_id["random_forest_correct"]
    ),
    "All_three_wrong": (
        ~comparison_by_id["llm_correct"] &
        ~comparison_by_id["xgboost_correct"] &
        ~comparison_by_id["random_forest_correct"]
    )
})


In [67]:
both_wrong_full_comparison = comparison_by_id[[
    "id",
    "label",
    "text_",
    "llm_correct",
    "xgboost_correct",
    "random_forest_correct"
]].copy()

both_wrong_full_comparison["LLM_XGBoost_wrong"] = (
    ~both_wrong_full_comparison["llm_correct"] &
    ~both_wrong_full_comparison["xgboost_correct"]
)

both_wrong_full_comparison["LLM_RandomForest_wrong"] = (
    ~both_wrong_full_comparison["llm_correct"] &
    ~both_wrong_full_comparison["random_forest_correct"]
)

both_wrong_full_comparison["XGBoost_RandomForest_wrong"] = (
    ~both_wrong_full_comparison["xgboost_correct"] &
    ~both_wrong_full_comparison["random_forest_correct"]
)

both_wrong_full_comparison["All_three_wrong"] = (
    ~both_wrong_full_comparison["llm_correct"] &
    ~both_wrong_full_comparison["xgboost_correct"] &
    ~both_wrong_full_comparison["random_forest_correct"]
)


In [63]:
# ============================================================
# Print review texts and labels where ALL THREE models are wrong
# ============================================================

all_three_wrong = comparison_by_id[
    ~comparison_by_id["llm_correct"] &
    ~comparison_by_id["xgboost_correct"] &
    ~comparison_by_id["random_forest_correct"]
].copy()

# Print label + review text
for i, row in all_three_wrong.iterrows():
    print(f"{i}. Label: {row['label']}")
    print(f"Text: {row['text_']}")
    print()

448. Label: CG
Text: For the kids who make this game, they love it.  We play it almost every day.

453. Label: CG
Text: These are good ones, I use them at the gym and I keep them for my kids. 

454. Label: OR
Text: They fit perfect, they look expensive, they are the most confortable shoes that i had ever. I love the design

475. Label: OR
Text: This product is sturdy and has a nice design.  Will last and last.

480. Label: OR
Text: I was very pleased with the quality of the video and sound, great concert, the Piano Guys are the best !!!



In [64]:
all_three_wrong[["label", "text_"]]

,label,text_
448,CG,"For the kids who make this game, they love it. We play it almost every day."
453,CG,"These are good ones, I use them at the gym and I keep them for my kids."
454,OR,"They fit perfect, they look expensive, they are the most confortable shoes that i had ever. I love the design"
475,OR,This product is sturdy and has a nice design. Will last and last.
480,OR,"I was very pleased with the quality of the video and sound, great concert, the Piano Guys are the best !!!"


| Pattern | Explanation |
|---|---|
| Personal-looking CG reviews | Some CG reviews mention personal use, kids, gym, or daily activity, so they look human-written. |
| Generic OR reviews | Some OR reviews are short, simple, and very positive, so they look fake or generated. |
| Little product-specific detail | The reviews do not contain enough concrete product information. |
| Surface-level signals | Models rely on phrases like “I use it”, “my kids”, “great quality”, or “nice design”, but these can appear in both CG and OR reviews. |
| Borderline cases | These examples are difficult because real and generated review styles overlap. |